In [1]:
#%pip install gtfs-kit movingpandas geopandas shapely keplergl

In [ ]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import gtfs_kit as gk
import movingpandas as mpd
from keplergl import KeplerGl

# 1. Load GTFS and select a representative service date
GTFS_PATH = "data/gtfs.zip"
feed = gk.read_feed(GTFS_PATH, dist_units="km")

week = feed.get_first_week()
service_date = week[0] if week else "20261123"
base_day = pd.to_datetime(service_date, format="%Y%m%d")

active_trips = feed.get_trips(date=service_date)
# Fallback if calendar.txt has no active service dates configured
if active_trips.empty:
    active_trips = feed.trips.copy()

active_trips = active_trips.merge(
    feed.routes[["route_id", "route_short_name", "route_long_name", "route_type"]],
    on="route_id",
    how="left",
)

/Library/Frameworks/Python.framework/Versions/3.15/lib/python3.15/site-packages/movingpandas/__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)


In [ ]:
# 2. Expand frequencies.txt into concrete vehicle trips
stop_times = feed.stop_times[
    feed.stop_times["trip_id"].isin(active_trips["trip_id"])
].copy()
stop_times = stop_times.sort_values(["trip_id", "stop_sequence"])

# Convert HH:MM:SS (supports GTFS hours >= 24) to Timedelta
stop_times["arr_td"] = pd.to_timedelta(stop_times["arrival_time"])
stop_times["dep_td"] = pd.to_timedelta(stop_times["departure_time"].fillna(stop_times["arrival_time"]))

expanded_trips = []
expanded_stop_times = []

has_freq = feed.frequencies is not None and not feed.frequencies.empty

for _, trip in active_trips.iterrows():
    tid = trip["trip_id"]
    st = stop_times[stop_times["trip_id"] == tid].copy()
    if len(st) < 2:
        continue

    trip_origin_td = st["dep_td"].iloc[0]
    st["rel_arr"] = st["arr_td"] - trip_origin_td
    st["rel_dep"] = st["dep_td"] - trip_origin_td

    freq_rows = (
        feed.frequencies[feed.frequencies["trip_id"] == tid]
        if has_freq
        else pd.DataFrame()
    )

    if not freq_rows.empty:
        # Generate individual departures across each headway window
        for f_idx, f_row in freq_rows.iterrows():
            start_td = pd.to_timedelta(f_row["start_time"])
            end_td = pd.to_timedelta(f_row["end_time"])
            headway = pd.to_timedelta(int(f_row["headway_secs"]), unit="s")

            dep_cursor = start_td
            seq_num = 0
            while dep_cursor <= end_td:
                new_tid = f"{tid}_f{f_idx}_{seq_num}"
                trip_copy = trip.copy()
                trip_copy["trip_id"] = new_tid
                expanded_trips.append(trip_copy)

                st_copy = st[["stop_id", "stop_sequence", "rel_arr", "rel_dep"]].copy()
                st_copy["trip_id"] = new_tid
                st_copy["timestamp"] = base_day + dep_cursor + st_copy["rel_arr"]
                expanded_stop_times.append(st_copy)

                dep_cursor += headway
                seq_num += 1
    else:
        # Standard fixed-timetable trip
        expanded_trips.append(trip)
        st_copy = st[["stop_id", "stop_sequence", "rel_arr"]].copy()
        st_copy["trip_id"] = tid
        st_copy["timestamp"] = base_day + st["arr_td"]
        expanded_stop_times.append(st_copy)

trips_df = pd.DataFrame(expanded_trips)
st_df = pd.concat(expanded_stop_times, ignore_index=True)

In [ ]:
# 3. Build shape geometries and interpolate vertex times
shapes_dict = {}
if feed.shapes is not None and not feed.shapes.empty:
    for shape_id, grp in feed.shapes.sort_values("shape_pt_sequence").groupby("shape_id"):
        coords = list(zip(grp["shape_pt_lon"], grp["shape_pt_lat"]))
        if len(coords) >= 2:
            shapes_dict[shape_id] = LineString(coords)

stops_lookup = feed.stops.set_index("stop_id")[["stop_lon", "stop_lat"]].to_dict("index")

point_records = []

for _, trip in trips_df.iterrows():
    tid = trip["trip_id"]
    route_label = (
        trip["route_short_name"]
        if pd.notna(trip["route_short_name"]) and str(trip["route_short_name"]).strip()
        else trip["route_long_name"]
    )
    st = st_df[st_df["trip_id"] == tid].sort_values("stop_sequence")
    if len(st) < 2:
        continue

    # Extract valid stop coordinates & epoch timestamps
    stop_pts = []
    stop_epochs = []
    for _, row in st.iterrows():
        s_info = stops_lookup.get(row["stop_id"])
        if s_info and pd.notna(row["timestamp"]):
            stop_pts.append(Point(s_info["stop_lon"], s_info["stop_lat"]))
            stop_epochs.append((row["timestamp"] - pd.Timestamp("1970-01-01")) / pd.Timedelta("1s"))

    if len(stop_pts) < 2:
        continue

    shape_line = shapes_dict.get(trip.get("shape_id"))
    if shape_line is None:
        shape_line = LineString([(p.x, p.y) for p in stop_pts])

    # Project stops onto the shape (0.0 to 1.0) and enforce monotonic progression
    stop_fractions = np.array([shape_line.project(p, normalized=True) for p in stop_pts])
    stop_fractions = np.maximum.accumulate(stop_fractions)
    stop_epochs = np.maximum.accumulate(np.array(stop_epochs))

    # Compute fractional position of every vertex along the road shape
    vertices = list(shape_line.coords)
    cum_dist = [0.0]
    for i in range(1, len(vertices)):
        dx = vertices[i][0] - vertices[i - 1][0]
        dy = vertices[i][1] - vertices[i - 1][1]
        cum_dist.append(cum_dist[-1] + np.hypot(dx, dy))

    total_len = cum_dist[-1]
    if total_len == 0:
        continue
    vertex_fractions = np.array(cum_dist) / total_len

    # Interpolate timestamp for every road vertex
    vertex_epochs = np.interp(vertex_fractions, stop_fractions, stop_epochs)
    vertex_times = pd.to_datetime(vertex_epochs, unit="s")

    for (lon, lat), ts in zip(vertices, vertex_times):
        point_records.append({
            "trip_id": tid,
            "route_name": str(route_label),
            "route_type": int(trip["route_type"]) if pd.notna(trip["route_type"]) else 3,
            "timestamp": ts,
            "geometry": Point(lon, lat),
        })

gdf_points = gpd.GeoDataFrame(point_records, crs="EPSG:4326")


In [ ]:
#gdf_points.to_file("gdf_points.json", driver="GeoJSON")

In [ ]:
# 4. Create & audit MovingPandas TrajectoryCollection
traj_collection = mpd.TrajectoryCollection(
    gdf_points, traj_id_col="trip_id", t="timestamp"
)

# Add speed (km/h) to spot or filter schedule anomalies (e.g., > 90 km/h)
traj_collection.add_speed(overwrite=True, units=("km", "h"))
print(f"Built {len(traj_collection)} vehicle trajectories.")

Built 29970 vehicle trajectories.


In [ ]:
# 5. Export to kepler.gl Trip Layer (4D [lon, lat, z, epoch])
features = []
for traj in traj_collection:
    df = traj.df
    if len(df) < 2:
        continue

    epochs = ((df.index - pd.Timestamp("1970-01-01")) // pd.Timedelta("1s")).tolist()
    coords_4d = [
        [round(pt.x, 6), round(pt.y, 6), 0, int(ep)]
        for pt, ep in zip(df.geometry, epochs)
    ]

    features.append({
        "type": "Feature",
        "properties": {
            "trip_id": traj.id,
            "route_name": df["route_name"].iloc[0],
            "route_type": int(df["route_type"].iloc[0]),
            "avg_speed_kmh": round(float(df["speed"].mean()), 1),
        },
        "geometry": {
            "type": "LineString",
            "coordinates": coords_4d,
        },
    })

trip_geojson = {"type": "FeatureCollection", "features": features}

# Save standalone interactive HTML (works reliably outside & inside Jupyter)
kepler_map = KeplerGl(height=700, data={"bus_movements": trip_geojson})
#kepler_map.save_to_html(file_name="current_network_movements.html")
#print("Saved animation to current_network_movements.html")

# If running inside JupyterLab/Notebook, display inline:
kepler_map